# Getting Started with Fully Sharded Data Parallel (FSDP2)

Source:

    https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html

- - -

## How FSDP2 works

In [DistributedDataParallel](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html) (DDP) training, each rank owns a model replica and processes a batch of data, finally it uses AllReduce to synchronize the gradients across ranks.

Comparing with DDP, FSDP reduces GPU memory footprint by sharding model weights, gradients, and optimizer state. It makes it feasible to train models that cannot fit on a single GPU. As shown below in the picture.

- Outside of forward and backward computation, weights are fully sharded.
- Before forward and backward, sharded weights are all-gathered into unsharded weights
- Inside backward, local unsharded gradients are reduce-scatterred into sharded gradients
- Optimizer updates sharded weights with sharded gradients, resulting in sharded optimizer state.

![FSDP workflow](img/fsdp_workflow.png)

FSDP can be considered a decomposition of DDP’s AllReduce operation into a ReduceScatter and an AllGather operations.

![FSDP all-gather and reduce-scatter](img/fsdp_sharding.png)

Comparing with [FSDP1](https://docs.pytorch.org/docs/stable/fsdp.html), FSDP2 has following advantages:

- Representing sharded weights as [DTensor](https://docs.pytorch.org/docs/stable/distributed.tensor.html) sliced over dimension `i`, allowing for easy manipulation of individual weights, communication-free sharded state dictionaries, and a simpler meta-device initialization flow.

- Improving memory management system that achieves lower and deterministic GPU memory by avoiding `recordStream` ([doc](https://dev-discuss.pytorch.org/t/fsdp-cudacachingallocator-an-outsider-newb-perspective/1486)) and does so without any CPU synchronization.

- Offering a tensor subclass extension point to customize the AllGather, e.g. for float8 AllGather for float8 Linear layers ([doc](https://dev-discuss.pytorch.org/t/enabling-float8-all-gather-in-fsdp2/2359)), and NF4 for QLoRA ([doc](https://github.com/pytorch/torchtune/blob/main/README.md))

- Mixing frozen and unfrozen weights can occur in the same communication group without using extra memory.

## How to use FSDP2

### Model Initialization

Applying `fully_shard` on submodules: Different from DDP, we should apply [fully_shard](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html) on submodules as well as the root model. In the transformer example below, we applied `fully_shard` on each layer first, then the root model

- During forward computation of `layers[i]`, the rest of the layers are sharded to reduce memory footprint.

- Inside `fully_shard(model)`, FSDP2 excludes weights from `model.layers` and classifies the remaining weights into a common weight group for performant AllGather and ReduceScatter operations.

- `fully_shard` moves sharded model to actual training device (for example, GPU).

**Command**: `torchrun --nproc_per_node 2 train.py`

In [ ]:
from torch.distributed.fsdp import fully_shard, FSDPModule

model = Transformer()
for layer in model.layers:
    fully_shard(layer)
fully_shard(model)

assert isinstance(model, Transformer)
assert isinstance(model, FSDPModule)
print(model)

"""
FSDPTransformer(
    (tok_embeddings): Embedding(...)
    ...
    (layers): 3 x FSDPTransformerBlock(...)
    (output): Linear(...)
  )
"""

We can inspect the nested wrapping with `print(model)`. `FSDPTransformer` is a joint class of [Transformer](https://github.com/pytorch/examples/blob/70922969e70218458d2a945bf86fd8cc967fc6ea/distributed/FSDP2/model.py#L100) and [FSDPModule](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.FSDPModule). The same thing happens to [FSDPTransformerBlock](https://github.com/pytorch/examples/blob/70922969e70218458d2a945bf86fd8cc967fc6ea/distributed/FSDP2/model.py#L76C7-L76C18). All FSDP2 public APIs are exposed through `FSDPModule`. For example, users can call `model.unshard()` to manually control AllGather schedules. See “explicit prefetching” below for details.

`model.parameters()` as DTensor: `fully_shard` shards weights across ranks, and convert `model.parameters()` from plain `torch.Tensor` to `DTensor` to represent sharded weights. FSDP2 slices tensors over dimension 0 by default, so `DTensor` placements are `Shard(dim=0)`. Say we have N ranks and a weight with N rows before sharding. After sharding, each rank will have 1 row of the weight. We can inspect sharded weights using `param.to_local()`.

In [ ]:
from torch.distributed.tensor import DTensor

for param in model.parameters():
    assert isinstance(param, DTensor)
    assert param.placements == (Shard(0),)
    # inspect sharded parameters with param.to_local()
    # print(f'{param.to_local()}')
    
optim = torch.optim.Adam(model.parameters(), lr=1e-2)

Note the optimizer is constructed after applying `fully_shard`. Both model and optimizer state dictionaries are represented in `DTensor`.

`DTensor` facilitates optimizer, gradient clipping and checkpointing:

- `torch.optim.Adam` and `torch.nn.utils.clip_grad_norm_` work out of the box with `DTensor` weights. This makes the code consistent between single-device and distributed training.

- we can use `DTensor` and DCP APIs to manipulate weights and get full state dictionaries, see “state dict” section below for details. For distributed state dictionaries, we can save/load checkpoints ([doc](https://docs.pytorch.org/tutorials/recipes/distributed_checkpoint_recipe.html)) without extra communication.

### Forward/Backward with Prefetching

**command**: `torchrun --nproc_per_node 2 train.py`

In [ ]:
for _ in range(epochs):
    x    = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    loss = model(x).sum()
    loss.backward()
    optim.step()
    optim.zero_grad()

 `fully_shard` registers forward/backward hooks to AllGather weights before computation, and reshards weights after computation. To overlap AllGathers with computation, FSDP2 offers implicit prefetching that works out of the box with the training loop above and explicit prefetching for advanced users to control AllGather schedules manually.

**Implicit Prefetching**: CPU thread issues AllGather `i` before layer `i`. AllGathers are queued into its own GPU stream while layer `i` computation happens in the default stream. For non-CPU-bound workload, for example, a Transformer with big batch size, AllGather `i+1` can overlap with computation for layer `i`. Implicit prefetching works similarly in the backward, except AllGathers are issued in the reverse of post-forward order.

![FSDP Implicit](img/fsdp_implicit.png)

We recommend users to start with implicit prefetching to understand the performance out of the box.

**Explicit Prefetching**: Users can specify forward ordering with [set_modules_to_forward_prefetch](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.FSDPModule.set_modules_to_forward_prefetch), and backward ordering with [set_modules_to_backward_prefetch](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.FSDPModule.set_modules_to_backward_prefetch). As shown in the code below, CPU thread issue AllGather `i + 1` and `i + 2` at layer `i`.

Explicit prefetching works well in following situation:

**CPU-bound workload**: If using implicit prefetching, CPU thread will be too slow to issue AllGather for layer `i+1` when kernels from layer `i` get executed. We have to explicitly issue AllGather `i+1` before running forward for layer `i`.

**Prefetching for 2+ layers**: Implicit prefetching only AllGathers next one layer at a time to keep memory footprint minimum. With explicit prefetching can all-gather multiple layers at a time to possibly for better performance with increased memory. See `layers_to_prefetch` in the code.

**Issuing 1st all-gather earlier**: Implicit prefetching happens at the time of calling `model(x)`. The first AllGather gets exposed. We can call `model.unshard()` explicitly earlier to issue 1st AllGather earlier.

**command**: `torchrun --nproc_per_node 2 train.py --explicit-prefetching`

In [ ]:
num_to_forward_prefetch = 2
for i, layer in enumerate(model.layers):
    if i >= len(model.layers) - num_to_forward_prefetch:
        break
    layers_to_prefetch = [
        model.layers[i + j] for j in range(1, num_to_forward_prefetch + 1)
    ]
    layer.set_modules_to_forward_prefetch(layers_to_prefetch)

num_to_backward_prefetch = 2
for i, layer in enumerate(model.layers):
    if i < num_to_backward_prefetch:
        continue
    layers_to_prefetch = [
        model.layers[i - j] for j in range(1, num_to_backward_prefetch + 1)
    ]
    layer.set_modules_to_backward_prefetch(layers_to_prefetch)

for _ in range(epochs):
    # trigger first AllGather earlier
    # this overlaps AllGather with any computation before forward pass 
    model.unshard()
    x    = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    loss = model(x).sum()
    loss.backward()
    optim.step()
    optim.zero_grad()

### Enabling Mixed Precision

FSDP2 offers a flexible [mixed precision policy](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.MixedPrecisionPolicy) to speed up training. One typical use case is:

- Casting float32 weights to bfloat16 for forward/backward computation, see `param_dtype=torch.bfloat16`
- Upcasting gradients to float32 for ReduceScatter to preserve accuracy, see `reduce_dtype=torch.float32`

Comparing with [torch.amp](https://docs.pytorch.org/docs/stable/amp.html), FSDP2 mixed precision has the following advantages:

- **Performant and flexible parameter casting**: All the weights inside a `FSDPModule` are cast together at the module boundary (before and after before/backward). We can set different mixed precision policies for each layer. For example, the first few layers can be in float32 while remaining layers can be in bfloat16.

- **float32 gradient reduction (ReduceScatter)**: Gradients might vary a lot from rank to rank. Reducing gradients to float32 can be critical for numeric stability.

**command**: `torchrun --nproc_per_node 2 train.py --mixed-precision`

In [ ]:
model = Transformer(model_args)

fsdp_kwargs = {
    "mp_policy": MixedPrecisionPolicy(
        param_dtype  = torch.bfloat16,
        reduce_dtype = torch.float32,
    )
}
for layer in model.layers:
    fully_shard(layer, **fsdp_kwargs)
fully_shard(model, **fsdp_kwargs)

# sharded weights are float32
for param in model.parameters():
    assert param.dtype == torch.float32

# unsharded weights are bfloat16
model.unshard()
for param in model.parameters(recurse=False):
    assert param.dtype == torch.bfloat16
model.reshard()

# optimizer state is in float32
optim = torch.optim.Adam(model.parameters(), lr=1e-2)

# training loop
# ...

### Gradient Clipping and Optimizer with DTensor

**command**: `torchrun --nproc_per_node 2 train.py`

In [ ]:
# 'optim' construction is based on DTensor model weights
optim = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(epochs):
    x    = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    loss = model(x).sum()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
    optim.step()
    optim.zero_grad()

Optimizer is initialized after applying `fully_shard` on the model, and holds reference to DTensor `model.parameters()`. For gradient clipping, `torch.nn.utils.clip_grad_norm_` works with DTensor weights. Tensor operations will be dispatched correctly inside DTensor to communicate partial tensors across ranks to preserve the single device semantic.

### State Dictionaries with DTensor APIs

We showcase how to convert a full state dictionary into a DTensor state dictionary for loading, and how to convert it back to full state dictionary for saving.

**command**: `torchrun --nproc_per_node 2 train.py`

For the first time, it creates checkpoints for the model and optimizer.

For the second time, it loads from the previous checkpoint to resume training.

**Loading state dictionaries**: We initialize the model under meta device and call `fully_shard` to convert `model.parameters()` from plain `torch.Tensor` to `DTensor`. After reading the full state dictionary from `torch.load`, we can call `distribute_tensor` to convert plain `torch.Tensor` into `DTensor`, using the same placements and device mesh from `model.state_dict()`. Finally we can call `model.load_state_dict` to load `DTensor` state dictionaries into the model.

In [ ]:
from torch.distributed.tensor import distribute_tensor

# mmap=True reduces CPU memory usage
full_sd = torch.load(
    "checkpoints/model_state_dict.pth",
    mmap         = True,
    weights_only = True,
    map_location = 'cpu',
)
meta_sharded_sd = model.state_dict()
sharded_sd      = {}

for param_name, full_tensor in full_sd.items():
    sharded_meta_param = meta_sharded_sd.get(param_name)
    sharded_tensor = distribute_tensor(
        full_tensor,
        sharded_meta_param.device_mesh,
        sharded_meta_param.placements,
    )
    sharded_sd[param_name] = nn.Parameter(sharded_tensor)

# `assign=True` since we cannot call `copy_` on meta tensor
model.load_state_dict(sharded_sd, assign=True)

**Saving state dictionaries**: `model.state_dict()` returns a `DTensor` state dictionary. We can convert a `DTensor` into a plain `torch.Tensor` by calling `full_tensor()`. Internally it issues an AllGather across ranks to get unsharded weights in plain `torch.Tensor`. For rank 0, `full_param.cpu()` offloads the tensor to CPU one by one to avoid peaking GPU memory with unsharded weights.

In [ ]:
sharded_sd     = model.state_dict()
cpu_state_dict = {}

for param_name, sharded_param in sharded_sd.items():
    full_param = sharded_param.full_tensor()
    if torch.distributed.get_rank() == 0:
        cpu_state_dict[param_name] = full_param.cpu()
    else:
        del full_param

torch.save(cpu_state_dict, "checkpoints/model_state_dict.pth")

Optimizer state dictionary works similarly ([code](https://github.com/pytorch/examples/blob/70922969e70218458d2a945bf86fd8cc967fc6ea/distributed/FSDP2/checkpoint.py#L156)). Users can customize the above `DTensor` scripts to work with 3rd party checkpoints.

If there is no need for customization, we can use [DCP APIs](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html) directly to support both single-node and multi-node training.

### State Dictionary with DCP APIs

**command**: `torchrun --nproc_per_node 2 train.py --dcp-api`

For the first time, it creates checkpoints for the model and optimizer.

For the second time, it loads from the previous checkpoint to resume training.

**Loading state dictionaries**: We can load a full state dictionary into a FSDP2 model with [set_model_state_dict](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.set_model_state_dict). With `broadcast_from_rank0=True`, we can load the full state dictionary only on rank 0 to avoid peaking CPU memory. DCP will shard tensors and broadcast them to other ranks.

In [ ]:
from torch.distributed.checkpoint.state_dict import set_model_state_dict

set_model_state_dict(
    model            = model,
    model_state_dict = full_sd,
    options          = StateDictOptions(
        full_state_dict      = True,
        broadcast_from_rank0 = True,
    ),
)

**Saving state dicts**: [get_model_state_dict](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.get_model_state_dict) with `full_state_dict=True` and `cpu_offload=True` AllGathers tensors and offload them to CPU. It works similarly to DTensor APIs.

In [ ]:
from torch.distributed.checkpoint.state_dict import get_model_state_dict

model_state_dict = get_model_state_dict(
    model   = model,
    options = StateDictOptions(
        full_state_dict = True,
        cpu_offload     = True,
    )
)

torch.save(model_state_dict, "model_state_dict.pth")

Refer to [pytorch/examples](https://github.com/pytorch/examples/blob/main/distributed/FSDP2/checkpoint.py) for loading and saving optimizer state dicts with [set_optimizer_state_dict](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.set_optimizer_state_dict) and [get_optimizer_state_dict](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.get_optimizer_state_dict).

## FSDP1-to-FSDP2 migration guide

Let us look at an example of an [FSDP](https://docs.pytorch.org/docs/stable/fsdp.html) usage and an equivalent [fully_shard](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html) usage. We will highlight the key differences and suggest steps for migration.

Original `FSDP()` usage:

Migration Steps:

- Replace the imports

- Implement our ‘policy’ directly (apply `fully_shard` to the desired sublayers)
- Wrap our root model with `fully_shard` instead of `FSDP`
- Get rid of `param_init_fn` and manually call `model.reset_parameters()`
- Replace other FSDP1 `kwargs` (see below)

**sharding_strategy**:

- FULL_SHARD: `reshard_after_forward=True`
- SHARD_GRAD_OP: `reshard_after_forward=False`
- HYBRID_SHARD: `reshard_after_forward=True` with a 2D device mesh
- _HYBRID_SHARD_ZERO2: `reshard_after_forward=False` with a 2D device mesh

**cpu_offload**:

- CPUOffload.offload_params=False: `offload_policy=None`
- CPUOffload.offload_params = True: `offload_policy=CPUOffloadPolicy()`

**backward_prefetch**:

- BACKWARD_PRE: always used
- BACKWARD_POST: not supported

**mixed_precision**:

- `buffer_dtype` is omitted because `fully_shard` does not shard buffers
- `fully_shard`’s `cast_forward_inputs` maps to both `cast_forward_inputs` and `cast_root_forward_inputs` in FSDP1
- `output_dtype` is a new configuration for `fully_shard`

**device_id**: Inferred from device_mesh’s device

**sync_module_states=True/False**: Moved to DCP. User can broadcast state dictionaries from rank0 using [`set_model_state_dict`](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.set_model_state_dict) with `broadcast_from_rank0=True`

**forward_prefetch**: Manual control over prefetching is possible with

- Manually call `fsdp_module.unshard()`
- Use these APIs to control automatic prefetching, [set_modules_to_forward_prefetch](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.FSDPModule.set_modules_to_forward_prefetch) and [set_modules_to_backward_prefetch](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.FSDPModule.set_modules_to_backward_prefetch)

**limit_all_gathers**: No longer needed, because `fully_shard` removed CPU synchronization

**use_orig_params**: Original weights are always used (no more flat parameter)

**no_sync()**: [set_requires_gradient_sync](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.FSDPModule.set_requires_gradient_sync)

**ignored_params** and **ignored_states**: [ignored_params](https://docs.pytorch.org/docs/main/distributed.fsdp.fully_shard.html#torch.distributed.fsdp.fully_shard)